# Time-decompression relay
Fires pulses of fluid through a PT column comprising pyDEW reactants.

In [1]:
import numpy as np
import pyDEW
from thermoengine import model
from copy import copy
from copy import deepcopy
import matplotlib.pyplot as plt
import pandas as pd
import linecache
import cmcrameri.cm as cmc
from scipy.interpolate import LinearNDInterpolator
from timeit import default_timer as timer
import warnings

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

# These are the additional functions which we need to work with pyDEW inputs/outputs
import pyDEW_functions as pDF


## A1. Preliminaries: mantle column
First we are going to establish the conditions of our column.

In [2]:
# This sets how many steps are taken from the base of the stack to the top
steps = 100 

# Create list of pressures
pressures = np.linspace(10, 15, steps+1)  # in kbar
pressures = pressures[::-1]  # invert for the temperatures (so we start at the base of the stack)
pressures = [i*1000 for i in pressures]  # in bar

# Create list of temperatures
temperatures = np.linspace(500, 600, steps+1)   # in degC
temperatures = [i+273.15 for i in temperatures]  # in K

Next we are going to set up our pyDEW system.

In [3]:
# Adjust here the basis species to be used in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2, placing CO3-2 directly where CO2(AQ) was to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)

# pyDEW minutia - to match Guillaume's setups
# Suppress H2CO3(AQ)
nxmods = [['H2CO3(AQ)', 0, -1, 0]]

# List of species to export - DEFUNCT
# exportSpecies = ['NA+', 'CA+2', 'MG+2', 'AL+3', 'H4SIO4(AQ)', 'H+', 'CO3-2', 'CL-', 'FE+2']

We will set our column to have the same composition throughout. This is going to be a list of list of reactants (which is a list of dictionaries). For each reaction, we are going to add a new list of reactants to the overall list.

In [4]:
# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
           'moles': 0.2,
           'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

spnDict = {'reactant': 'SPINEL',
           'moles': 0.04}

startingReactants = [olvDict, opxDict, cpxDict, spnDict]

# Quickly check that our starting fluid-rock ratio (F/R) is going to be 1 (or whatever it is)
startingReactants = pDF.reactantToFR(startingReactants, 1.0, dew_system)

# Create our preliminary compositional stack
startingComposition = pDF.reactantsToDF(startingReactants, dew_system)
startingComposition = pd.concat([startingComposition]*len(pressures))
startingComposition['Pressure (bars)'] = pressures
startingComposition['Temperature (K)'] = temperatures
startingComposition['Time'] = [0]*len(pressures)

## A2. Fluid composition
Import the converged EQ3 solutions of G. Siron. This will be the basis for interpolation to get slab fluid compositions.

Create our input fluid and fO2:

In [5]:
# New function here to find a fluid from the database which falls in our 3D space.
def createHybridFluid(proportions, T, P, database='Pickups_molalities_fO2_kw.xlsx',
                      sheetNames=['DMM','MORB','Sediments']):
    """
    Function to create a hybrid linearly interpolated fluid from fluid endmember compositions. Fluid assumed to be
    linearly proportional of endmembers.
    
    Inputs
    ------
    proportions : list of floats
        proportions each fluid endmember contributes to the final fluid
    T : float
        temperature for interpolation, in K
    P : float
        pressure for interpolation, in bars
    database : string
        xlsx database with fluid endmember compositions. Default is 'Pickups_molalities_fO2_kw.xlsx'
    sheetNames : list of strings
        names of sheets in xlsx database corresponding to fluid endmembers. Default is ['DMM','MORB','Sediments'],
        corresponding to 'Pickups_molalities_fO2_kw.xlsx'
    
    Outputs
    -------
    exportDict : dictionary
        dictionary describing elemental composition of hybridised fluid
    exportfO2 : float
        float describing hybridised oxygen fugacity
    """
    
    assert len(proportions) == len(sheetNames), 'check same item proportions and database'
    
    for i in range(len(proportions)):
        
        # To store our new fluid
        if i == 0:
            molalitiesDict = {}
        
        key = sheetNames[i]
        LithData = pd.read_excel(database, sheet_name=sheetNames[i])
        proportion = proportions[i]/np.sum(proportions)
        
        # Find the species comprising the fluid
        columnHeaders = list(LithData.columns)
        speciesHeaders = columnHeaders[2:]
        
        # Extract pressures and temperatures
        dataPressures = LithData['P (kbar)'].values
        dataTemperatures = LithData['T (degC)'].values
        
        # This iterates over all our species and linearly interpolates across them all based on the results
        # from the molalities pickup sheet
        for j in range(len(speciesHeaders)):
            speciesData = LithData[speciesHeaders[j]]

            # Makes sure that the species aren't all constant, which makes the interpolation wacky
            if speciesData.tolist() == [speciesData.tolist()[0]]*len(speciesData):
                if i == 0:
                    molalitiesDict[speciesHeaders[j]] = speciesData[0] * proportion
                else:
                    molalitiesDict[speciesHeaders[j]] = molalitiesDict[speciesHeaders[j]] + (speciesData[0] * proportion)

            # Otherwise we set up the interpolator for the species given pressure and temperature
            else:
                interp = LinearNDInterpolator(list(zip(dataTemperatures.tolist(),
                                                       dataPressures.tolist())),
                                              speciesData.tolist())

                # key to reconvert back to original units here
                interpMolal = interp(T-273.15, P/1000)  
                if i == 0:
                    molalitiesDict[speciesHeaders[j]] = float(interpMolal) * proportion
                elif speciesHeaders[j] not in molalitiesDict.keys():
                    molalitiesDict[speciesHeaders[j]] = float(interpMolal) * proportion
                else:
                    molalitiesDict[speciesHeaders[j]] = molalitiesDict[speciesHeaders[j]] + (float(interpMolal) * proportion)
        
    exportDict = {'NA+': molalitiesDict['NA'],
                  'K+' : molalitiesDict['K'],
                  'CA+2': molalitiesDict['CA'],
                  'MG+2': molalitiesDict['MG'],
                  'AL+3': molalitiesDict['AL'],
                  'H4SIO4(AQ)': molalitiesDict['SI'],
                  'CO3-2' : molalitiesDict['C'],
                  'CL-': molalitiesDict['CL'],
                  'FE+2': molalitiesDict['FE']}
    exportfO2 = molalitiesDict['fO2'] 
    return exportDict, exportfO2

# Test
createHybridFluid([1, 1, 1], 500+273.15, 10*1000)

({'NA+': 0.5112796036061126,
  'K+': 0.002554713917784626,
  'CA+2': 0.0018076691611750852,
  'MG+2': 0.35464197510929396,
  'AL+3': 0.015893954480061458,
  'H4SIO4(AQ)': 0.1456368823323797,
  'CO3-2': 0.0010000001358244134,
  'CL-': 0.010000000008220366,
  'FE+2': 0.008087005657494793},
 -22.137493068396594)

In [6]:
# Get starting PT for interpolation
sourcePressure = pressures[0]
sourceTemperature = temperatures[0]


## B. Time-decompression relay function

In [7]:
def decompressionRelay(startingColumn, startingMolalities, startingfO2, system, startingFRRatio=1.0, timesteps=10,
                       exportPath='data/bigtest', outputToRead='working/output', tabToRead='working/tab',
                       FRRatioOption0=None, FRRatioOption1=None):
    """
    Decompression relay as a function of time. This models a column of rock getting flushed with a fluid periodically.
    Definitions:
    - initial : before the reaction at each stage, i.e., relative to 1 kg fluid
    - final : after the reaction at each stage, i.e., relative to 1 kg fluid
    - running : relative to the overall fluid
    
    INPUTS
    ------
    startingColumn : pandas.DataFrame
        pandas DataFrame with each column a different mineral concentration (moles), with two additional columns
        'Pressure (bars)' and 'Temperature (K)'
    startingMolalities : dictionary
        dictionary of elemental molalities in log molal describing the composition of the flushing fluid
    system : pyDEW.System object
        pyDEW.System for calculations
    timesteps : integer
        number of times to flush our solid with the fluid
    exportPath : string
        path to export xlsx to
    outputToRead : string
        path to pyDEW.Reaction output file
    tabToRead : string
        path to pyDEW.Reaction tab file
    FRRatioOption0 : None, 'fluid', 'fixed', 'limit'
        first batch of options for fluid-rock ratio
    FRRatioOption1 : float
        fluid-rock ratio (dependent on FRRatioOption0)

    OUTPUTS
    -------
    xlsx file
        Excel file with solid, element, and fluid speciation data
        
    """

    start = timer()
    t = 0
    crashFlag = False
    
    # The headers for our output file may contain solid-solution minerals and their endmembers
    # We need to set the headers up now so we can allow for the mineral endmembers and some other options
    acceptableHeaders = [i.props['phase_name'] for i in system.minerals]
    acceptableHeaders.extend(['Pressure (bars)', 'Temperature (K)', 'Time', 'max log zi',
                              'Mass (g)', 'Volume (cc)', 'Density (g/cc)', 'Crash flag'])
    acceptableHeaders.extend(['F/R before reaction', 'F/R after reaction', 'Running F/R', 'Initial mass of solid',
                              'Initial mass of fluid', 'Final mass of solid', 'Final mass of fluid'])
    acceptableHeaders.extend(['DIOPSIDE(CPX)', 'JADEITE(CPX)', 'HEDENBERGITE(CPX)'])
    acceptableHeaders.extend(['KFELDSPAR(KFSP)', 'ALBITE(KFSP)'])

    ##### TIME LOOP #####
    print('Time loop started')
    while t < timesteps:
        if t == 0:
            workingColumn = startingColumn
        else:
            workingColumn = lastColumn
        
        
        # Storage for running F/R throughout column, and for tracking solid masses and volumes
        initialFRRatioList = []  # This is keeping track of the F/R before each stage of reaction
        finalFRRatio1kgList = []  # This is keeping track of the F/R after each reaction relative to 1 kg fluid
        runningFRRatioList = []  # This is keeping track of the F/R after each stage of reaction

        # solidMass = []  # This is keeping track of the mass of the solid after the reactions
        
        initialSolidMassList = []  # This is keeping track of the mass of the solid before the reactions
        initialFluidMassList = []  # This is keeping track of the mass of the fluid before the reactions

        finalFluidMassList = []  # This is keeping track of the mass of the fluid after the reactions
        finalSolidMassList = []  # This is keeping track of the mass of the solid after the reactions
        
        
        # solidVolume = []  # This is keeping track of the volume of the solid after the reactions
        
        # This is for keeping track of crashes that will trigger the UserError try clause below
        crashFlag = False
        
        ##### DECOMPRESSION RELAY #####
        for n in range(len(startingColumn)):
            #print(n,t)
            
            #### ESTABLISH REACTANT CONDITIONS ####
            workingParameters = workingColumn.iloc[n]
            workingReactants = pDF.DFtoReactant(workingParameters, system)
            workingReactants = pDF.removeNaNReactants(workingReactants)
            workingT = workingParameters['Temperature (K)']
            workingP = workingParameters['Pressure (bars)']
            
            #######################################################################################################################
            
            #### ESTABLISH FLUID CONDITIONS ####
            # If first, then use the slab fluid. Else use the fluid from the previous calculation
            if n == 0:
                workingFluid = pyDEW.Fluid(system, workingT, workingP, molalities=startingMolalities, fO2=startingfO2,
                                           uebal='H+', uacion='CL-', nxmods=nxmods, aH2O_mode='molfraction')
                
                # Also adjust our working reactants by the appropriate F/R
                workingReactants = pDF.reactantToFR(workingReactants, startingFRRatio, system)
                
            else:
                # If no mineral product was found previously, use the previous lastElements/workingfO2 (i.e., crashFlag=True)
                # Else, update with the values from the calculation prior to the last one (i.e., crashFlag=False)
                if crashFlag == False:
                    lastElements = dewReaction.elements.iloc[-1]
                    workingfO2 = dewReaction.fO2[-1]

                molalities = {'NA+': lastElements['Na'],
                              'CA+2': lastElements['Ca'],
                              'MG+2': lastElements['Mg'],
                              'AL+3': lastElements['Al'],
                              'H4SIO4(AQ)': lastElements['Si'],
                              'CO3-2' : lastElements['C'],
                              'CL-': lastElements['Cl'],
                              'FE+2': lastElements['Fe']}
                molalities.update((i, 10**j) for i, j in molalities.items()) # All reported as log(molal) in pyDEW output
                workingFluid = pyDEW.Fluid(system, workingT, workingP, molalities=molalities, fO2=workingfO2,
                                           uebal='H+', uacion='CL-', nxmods=nxmods, aH2O_mode='molfraction')
                
            #######################################################################################################################
            
            #### MODIFY FLUID-ROCK RATIO OPTIONS ####
            # Modify reactant to maintain/change fluid-rock ratio if needed. If None, then 1 kg of fluid at each step
            # This is still a work in progress while I make sure the skeleton of the relay works
            # solidModulate is the value to multiply the moles of output minerals by.
            # The initial mass of rock is multiplied by a value so that the amount of fluid at 1 kg interacts with an
            # appropriate amount of rock given a starting F/R. By using the same value on the products, we can determine
            # the mass of solid remaining after the reaction. 
            # e.g., if 1 kg of rock interacts with 0.5 kg of fluid, then the initial calculation is 2 kg of rock with
            # 1 kg of fluid. After, we may end up with, e.g., 2.1 kg of resultant rock and 0.9 kg fluid. We will correct this
            # by applying the same coefficient to end up with 1.05 kg of resultant rock and 0.45 kg fluid.
            # solidModulate is the amount to multiply our product by to return back to the correct amount of solid
            
            # If None, then the fluid will remain at 1 kg for every step of the reaction
            # In practice this will make the F/R increase as the relay progresses (unrealistic)
            if FRRatioOption0 == None:
                solidModulate = 1.0
            
            # If 'fluid', then the fluid will decreases with each progressive step as it is consumed, so we need to adjust our solid
            # so that it is appopriate for 1 kg fluid. Here we need to track the running F/R. massShift is the ratio between final
            # mass of fluid and initial mass of fluid
            elif FRRatioOption0 == 'fluid':
                
                # If starting, then the initialFRRatio will be whatever is provided as an input
                if n == 0:
                    massShift = 1.0
                    # runningInitialFRRatio = startingFRRatio
                else:
                    # This is the fluid mass change from the last reaction - this is found from the EQ6 output
                    # If there is a crash then use the previous value
                    if crashFlag:
                        massShift = 1.0
                    else:
                        # Mass shift here is the ratio of final to initial of fluid (i.e., relative to 1 kg of EQ6 fluid)
                        massShift = pDF.findFluidMassChange(tabToRead)
                #     runningInitialFRRatio = runningInitialFRRatio * massShift
                # runningInitialFRRatios.append(runningInitialFRRatio)
                
                # For each mineral in the reactant, divide the moles by the mass shift to simulate a change in fluid mass
                # which reflects the 'true' F/R based on the initial F/R. As the F/R is expected to decrease with ongoing reaction,
                # the number of moles of solid is expected to increase to emulate the consumption of solid.
                # Remember - this is based on the previous calculation.
                for j in range(len(workingReactants)):
                    workingReactants[j]['moles'] = workingReactants[j]['moles'] / massShift
                    
                # workingReactants is now relative to 1 kg of fluid
                solidModulate = massShift

            # FRRatio is the initial mass of fluid divided by the mass of reactants
            # stageInitialFRRatio is the F/R at that given n before the reaction
            initialFRRatio = 1000.0 / pDF.findReactantMass(workingReactants, system)
            
            # Keep track of the mass of fluid and solid before the reaction around
            if n == 0:
                assert abs(1 - (initialFRRatio / startingFRRatio)) < 0.001, 'check starting FRR, ' + str(initialFRRatio) + ' v ' + str(startingFRRatio)
                initialFluidMass = 1000.0
                runningFRRatio = startingFRRatio
            
            # If we have had a crash then we need to use the values from the previous run
            if crashFlag == False:
                initialFluidMass = initialFluidMass * initialFRRatio

            initialFRRatioList.append(initialFRRatio)
            initialFluidMassList.append(initialFluidMass)
            initialSolidMassList.append(pDF.findReactantMass(workingReactants, system) / massShift)
            
            #######################################################################################################################

            #### RUN REACTION ####
            # Calculate the reaction here between reactants and products. The output will be a series of pandas dataframes
            try:
                # If the reaction goes ahead without issue, then perform the reaction with the given conditions
                dewReaction = pyDEW.Reaction(workingFluid, workingT, workingP, workingReactants, zimax=5.0, calculation_mode='titration')
                
                #### CREATE NEW DATAFRAMES FOR OUTPUT ####
                # Get the solids formed during this reaction (from 1 kg fluid)
                # This doesn't work if there is no solid product, which is why this is all nestled in a big try clause
                lastReactionDF = pDF.findSolidComposition(outputToRead, system, workingP, workingT)
                maxLogZi = lastReactionDF['log zi'].iloc[-1]  # Get maximum reaction progress
                
                # Get the dataframe representing the last solid (from 1 kg fluid)
                lastSolid = lastReactionDF[lastReactionDF['log zi'] == maxLogZi]
                lastSolidHeaders = lastSolid['phase/end-member'].tolist()
                lastSolidValues = lastSolid['moles'].tolist()

                # Because CPX has two different solid solutions, need to clarify the differences or pandas throws a fit
                if 'CLINOPYROXENE(SS)' in lastSolidHeaders:
                    index = lastSolidHeaders.index('CLINOPYROXENE(SS)')
                    lastSolidHeaders[index+1] = 'DIOPSIDE(CPX)'
                    lastSolidHeaders[index+2] = 'JADEITE(CPX)'
                    lastSolidHeaders[index+3] = 'HEDENBERGITE(CPX)'

                # Same thing for k-spar
                if 'KFELDSPAR(SS)' in lastSolidHeaders:
                    index = lastSolidHeaders.index('KFELDSPAR(SS)')
                    lastSolidHeaders[index+1] = 'KFELDSPAR(KFSP)'
                    lastSolidHeaders[index+2] = 'ALBITE(KFSP)'

                # Create new dataframe and reactant object for our final reactant (from 1 kg fluid)
                lastSolidDF = pd.DataFrame([lastSolidValues])
                lastSolidDF.columns = lastSolidHeaders
                lastSolidDF = lastSolidDF[lastSolidDF.columns.intersection(acceptableHeaders)]  # Remove mineral names with solid solution

                # There is an issue where one of the columns is nearly zero and when the dataframes are concated together all columns
                # duplicate except for the one which is nearly zero which is read as a new zero column. This code should make sure that
                # column headers are not repeated.
                if len(lastSolidDF.columns) != len(set(lastSolidDF.columns)):
                    lastSolidDF = lastSolidDF.iloc[:, ~lastSolidDF.columns.duplicated()]

                # Convert our pandas output into a reactant list of dictionaries
                lastSolidReactant = pDF.DFtoReactant(lastSolidDF.iloc[0], system)

                # Determine the physical properties of the reactant at the current PT
                lastSolidMass = pDF.findReactantMass(lastSolidReactant, system)
                lastSolidVolume = pDF.findReactantVolume(lastSolidReactant, system, workingP, workingT)
                lastSolidDensity = lastSolidMass / lastSolidVolume
                
                # So far everything has been done relative to 1 kg of fluid. Here we need to recalculate the fluid and solid masses.
                # Get the final mass of fluid and solid
                lastFluidMass = 1000.0 * pDF.findFluidMassChange(tabToRead)  # Mass of the fluid at the end of the reaction (relative to initial 1 kg)
                finalFRRatio = lastFluidMass / lastSolidMass
                
                finalFluidMassList.append(lastFluidMass / solidModulate)  # This is the 'true' mass of the fluid
                finalSolidMassList.append(lastSolidMass / solidModulate)  # This is the 'true' mass of the solid
                finalFRRatio1kgList.append(finalFRRatio)  # This is the F/R after this stage of reaction
                
                runningFRRatio = runningFRRatio * finalFRRatio
                runningFRRatioList.append(runningFRRatio)

                # Multiply the product solid values by the modulate value to get the true solid mass (i.e., for whatever
                # amount of fluid there is before correction to 1 kg for F/R). Then do the same as above for each corrected solid
                adjustedLastSolidReactant = []
                for j in lastSolidReactant:
                    j['moles'] = solidModulate * j['moles']
                    adjustedLastSolidReactant.append(j)
                adjustedLastSolidDF = pDF.reactantsToDF(adjustedLastSolidReactant, system)
                adjustedLastSolidDF.reset_index().T.drop_duplicates().T
                adjustedLastSolidMass = pDF.findReactantMass(adjustedLastSolidReactant, system)
                adjustedLastSolidVolume = pDF.findReactantVolume(adjustedLastSolidReactant, system, workingP, workingT)
                adjustedLastSolidDensity = adjustedLastSolidMass / adjustedLastSolidVolume

                # Set up the information for the fluid of the last reaction
                lastReactionElementsHeaders = dewReaction.elements.columns
                lastReactionElements = dewReaction.elements.iloc[-1].tolist()
                lastReactionSpeciesHeaders = dewReaction.species_concs.columns
                lastReactionSpecies = dewReaction.species_concs.iloc[-1].tolist()

                # Save the data for the next iteration in case it fails
                backupSolidDF = deepcopy(lastSolidDF)
                backupAdjustedSolidDF = deepcopy(adjustedLastSolidDF)
                backupElements = deepcopy(lastReactionElements)
                backupFluid = deepcopy(lastReactionSpecies)

                # Undo the crashFlag if it has been triggered
                crashFlag = False
                
            #######################################################################################################################
                    
            except KeyError:
                # If the resultant dataframe is empty, then no reaction has happened and we should use the previous input as the output
                print('Calculation at [n=' + str(n) +', t=' + str(t) + '] has failed: using results from previous n')
                
                # This is in case there is no reaction at the start - this has not been encountered yet so I don't know if this works
                if n == 0:
                    print('Calculation at n=0 has failed; retaining solid composition and moving fluid along')
                    lastSolidDF = pDF.reactantsToDF(workingReactants, system)
                    adjustedLastSolidReactant = []
                    for j in workingReactants:
                        j['moles'] = solidModulate * j['moles']
                        adjustedLastSolidReactant.append(j)
                    adjustedLastSolidDF = pDF.reactantsToDF(adjustedLastSolidReactant, system)
                    
                    # Get the elements and reformat
                    lastReactionElementsHeaders = workingFluid.elemental_comp['element'].tolist()
                    lastReactionElementMolal = workingFluid.elemental_comp['molality'].tolist()
                    lastReactionElementLogMolal = [[np.log10(i)] for i in lastReactionElementMolal]
                    lastReactionElements = pd.DataFrame(lastReactionElementLogMolal)
                    lastReactionElements.columns = lastReactionElementsHeaders
                    
                    # Get the species and reformat
                    lastReactionSpeciesHeaders = workingFluid.aqueous_species['species'].tolist()
                    lastReactionSpeciesMolal = workingFluid.aqueous_species['molality'].tolist()
                    lastReactionSpeciesLogMolal = [[np.log10(i)] for i in lastReactionSpeciesMolal]
                    lastReactionSpecies = pd.DataFrame(lastReactionSpeciesLogMolal)
                    lastReactionSpecies.columns = lastReactionSpeciesHeaders
                    
                    # Get the physical properties of the solid
                    lastSolidMass = findReactantMass(workingReactants, system)
                    lastSolidVolume = findReactantVolume(workingReactants, system, workingP, workingT)
                    lastSolidDensity = lastSolidMass / lastSolidVolume
                    
                # Otherwise, we can use our backups - the physical properties are already calculated
                else:
                    lastSolidDF = backupSolidDF
                    adjustedLastSolidDF = backupAdjustedSolidDF
                    lastReactionElements = backupElements
                    lastReactionSpecies = backupFluid
                
                # Append some things to the lists which otherwise would not be appended
                finalFRRatio1kgList.append(finalFRRatio1kgList[-1])
                finalFluidMassList.append(finalFluidMassList[-1])
                finalSolidMassList.append(finalSolidMassList[-1])
                runningFRRatioList.append(runningFRRatioList[-1])
                
                # Flip the flag up so the code recognises that a crash has happened and can react accordingly above
                crashFlag = True

            # Create DataFrames for solid, fluid elements, fluid species
            if n == 0:
                # This should be the solid at the given stage/time
                timeSolidDF = deepcopy(lastSolidDF)
                timeSolidDF['Pressure (bars)'] = [workingP]
                timeSolidDF['Temperature (K)'] = [workingT]
                timeSolidDF['Time'] = [t + 1]
                timeSolidDF['max log zi'] = [maxLogZi]
                timeSolidDF['Mass (g)'] = [lastSolidMass]
                timeSolidDF['Volume (cc)'] = [lastSolidVolume]
                timeSolidDF['Density (g/cc)'] = [lastSolidDensity]
                timeSolidDF['Crash flag'] = [crashFlag]
                
                # This should be adjusted for total mass of the rock
                allSolidDF = deepcopy(adjustedLastSolidDF)
                allSolidDF['Pressure (bars)'] = [workingP]
                allSolidDF['Temperature (K)'] = [workingT]
                allSolidDF['Time'] = [t + 1]
                allSolidDF['max log zi'] = [maxLogZi]
                allSolidDF['Mass (g)'] = [adjustedLastSolidMass]
                allSolidDF['Volume (cc)'] = [adjustedLastSolidVolume]
                allSolidDF['Density (g/cc)'] = [adjustedLastSolidDensity]
                
                timeElementsDF = pd.DataFrame([lastReactionElements])
                timeElementsDF.columns = lastReactionElementsHeaders
                timeElementsDF['Pressure (bars)'] = [workingP]
                timeElementsDF['Temperature (K)'] = [workingT]
                timeElementsDF['Time'] = [t + 1]

                timeFluidDF = pd.DataFrame([lastReactionSpecies])
                timeFluidDF.columns = lastReactionSpeciesHeaders
                timeFluidDF['Pressure (bars)'] = [workingP]
                timeFluidDF['Temperature (K)'] = [workingT]
                timeFluidDF['Time'] = [t + 1]
            
            # Else append the new data to the previous step
            else:
                tempSolidDF = deepcopy(lastSolidDF)
                tempSolidDF['Pressure (bars)'] = [workingP]
                tempSolidDF['Temperature (K)'] = [workingT]
                tempSolidDF['Time'] = [t + 1]
                tempSolidDF['max log zi'] = [maxLogZi]
                tempSolidDF['Mass (g)'] = [lastSolidMass]
                tempSolidDF['Volume (cc)'] = [lastSolidVolume]
                tempSolidDF['Density (g/cc)'] = [lastSolidDensity]
                tempSolidDF['Crash flag'] = [crashFlag]
                timeSolidDF = pd.concat([timeSolidDF, tempSolidDF])
                
                tempAllSolidDF = deepcopy(adjustedLastSolidDF)
                tempAllSolidDF['Pressure (bars)'] = [workingP]
                tempAllSolidDF['Temperature (K)'] = [workingT]
                tempAllSolidDF['Time'] = [t + 1]
                tempAllSolidDF['max log zi'] = [maxLogZi]
                tempAllSolidDF['Mass (g)'] = [adjustedLastSolidMass]
                tempAllSolidDF['Volume (cc)'] = [adjustedLastSolidVolume]
                tempAllSolidDF['Density (g/cc)'] = [adjustedLastSolidDensity]

                allSolidDF = pd.concat([allSolidDF, tempAllSolidDF])

                tempElementsDF = pd.DataFrame([lastReactionElements])
                tempElementsDF.columns = lastReactionElementsHeaders
                tempElementsDF['Pressure (bars)'] = [workingP]
                tempElementsDF['Temperature (K)'] = [workingT]
                tempElementsDF['Time'] = [t + 1]
                timeElementsDF = pd.concat([timeElementsDF, tempElementsDF])

                tempFluidDF = pd.DataFrame([lastReactionSpecies])
                tempFluidDF.columns = lastReactionSpeciesHeaders
                tempFluidDF['Pressure (bars)'] = [workingP]
                tempFluidDF['Temperature (K)'] = [workingT]
                tempFluidDF['Time'] = [t + 1]
                timeFluidDF = pd.concat([timeFluidDF, tempFluidDF])
                
            
        #### REMOVE BAD HEADERS ####
        # The solid-solution series may be recorded, in which case we remove the mineral in question and use the endmembers
        timeSolidDF = timeSolidDF[timeSolidDF.columns.intersection(acceptableHeaders)]
        timeSolidDF['F/R before reaction'] = initialFRRatioList
        timeSolidDF['F/R after reaction'] = finalFRRatio1kgList
        timeSolidDF['Running F/R'] = runningFRRatioList
        timeSolidDF['Initial mass of solid'] = initialSolidMassList
        timeSolidDF['Initial mass of fluid'] = initialFluidMassList
        timeSolidDF['Final mass of solid'] = finalSolidMassList
        timeSolidDF['Final mass of fluid'] = finalFluidMassList
        
        #### COMBINE TIMESTEP DATAFRAME WITH TOTAL DATAFRAME ####
        if t == 0:
            totalSolidDF = timeSolidDF
            totalAllSolidDF = allSolidDF 
            totalElementsDF = timeElementsDF
            totalFluidDF = timeFluidDF
        else:
            totalSolidDF = pd.concat([totalSolidDF, timeSolidDF])
            totalAllSolidDF = pd.concat([totalAllSolidDF, allSolidDF])
            totalElementsDF = pd.concat([totalElementsDF, timeElementsDF])
            totalFluidDF = pd.concat([totalFluidDF, timeFluidDF])
        lastColumn = deepcopy(timeSolidDF)
        print('Timestep t=' + str(t) + ' is complete, with ' + str(n*(t+1)) + ' total calculations performed')
        t+=1
    
    #### EXPORT ####
    # Export to xlsx
    with pd.ExcelWriter(exportPath+'.xlsx') as writer:
        totalSolidDF.to_excel(writer, sheet_name='solid')
        totalAllSolidDF.to_excel(writer, sheet_name='solid2')
        totalElementsDF.to_excel(writer, sheet_name='elements')
        totalFluidDF.to_excel(writer, sheet_name='fluid')
    print('data exported to: ' + exportPath+'.xlsx')
    end = timer()
    print('Total calculations: ' + str(timesteps*len(startingColumn)))
    print('Total runtime: ' + str(round(end-start, 1)) + ' seconds')
    print('Total runtime: ' + str(round((end-start)/60, 1)) + ' minutes')
    return


In [ ]:
startingFluid = createHybridFluid([1, 0, 0], sourceTemperature, sourcePressure)
decompressionRelay(startingComposition, startingFluid[0], startingFluid[1],
                   dew_system, timesteps=100,
                   exportPath='data/DMM_P15-10_T500-600_n100_t100', FRRatioOption0='fluid')

Time loop started
Timestep t=0 is complete, with 100 total calculations performed
Timestep t=1 is complete, with 200 total calculations performed
Timestep t=2 is complete, with 300 total calculations performed
Timestep t=3 is complete, with 400 total calculations performed
Timestep t=4 is complete, with 500 total calculations performed
Timestep t=5 is complete, with 600 total calculations performed
Timestep t=6 is complete, with 700 total calculations performed
Timestep t=7 is complete, with 800 total calculations performed
Timestep t=8 is complete, with 900 total calculations performed
Timestep t=9 is complete, with 1000 total calculations performed
Timestep t=10 is complete, with 1100 total calculations performed
Timestep t=11 is complete, with 1200 total calculations performed
Timestep t=12 is complete, with 1300 total calculations performed
Timestep t=13 is complete, with 1400 total calculations performed
Timestep t=14 is complete, with 1500 total calculations performed
Timestep t=

In [ ]:
startingFluid = createHybridFluid([0, 1, 0], sourceTemperature, sourcePressure)
decompressionRelay(startingComposition, startingFluid[0], startingFluid[1],
                   dew_system, timesteps=100,
                   exportPath='data/MORB_P15-10_T500-600_n100_t100', FRRatioOption0='fluid')

In [ ]:
startingFluid = createHybridFluid([0, 0, 1], sourceTemperature, sourcePressure)
decompressionRelay(startingComposition, startingFluid[0], startingFluid[1],
                   dew_system, timesteps=100,
                   exportPath='data/SED_P15-10_T500-600_n100_t100', FRRatioOption0='fluid')

In [ ]:
startingFluid = createHybridFluid([1, 1, 1], sourceTemperature, sourcePressure)
decompressionRelay(startingComposition, startingFluid[0], startingFluid[1],
                   dew_system, timesteps=100,
                   exportPath='data/EQ_P15-10_T500-600_n100_t100', FRRatioOption0='fluid')